# DATA CLEANING

This notebook shows in details the data cleaning phase related to the bitcoin's volatility analysis.  
  
input: raw dataset, "crypto_volatility.csv"  
output: cleaned and ready-to-be-analyzed df, "cryptoVol_clean.csv"  

## FUNCTIONS

- Duplicates checking

In [16]:
def find_duplicates(df):
    """
    Identifies duplicate rows in a DataFrame where all column values are identical
    and sorts them to appear one below the other for easy comparison.

    Parameters:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A DataFrame where duplicates appear contiguously.
    """
    # Find all duplicate rows (including first occurrence)
    duplicate_mask = df.duplicated(keep=False)

    # Extract duplicates and sort them
    duplicates_df = df[duplicate_mask].sort_values(by=list(df.columns)).reset_index(drop=True)

    return duplicates_df

- Outliers Checking (Z-score and interquartile method)

In [21]:
#Define a function to check for outliers with IQR range or Z-score
def count_outliers(df, method='iqr', threshold=3):
    outlier_counts = {}

    for column in df.select_dtypes(include=[np.number]).columns:  # Only consider numeric columns
        if method == 'iqr':
            # IQR method
            Q1 = df[column].quantile(0.25)
            Q3 = df[column].quantile(0.75)
            IQR = Q3 - Q1
            outliers = ((df[column] < (Q1 - 1.5 * IQR)) | (df[column] > (Q3 + 1.5 * IQR)))
        elif method == 'zscore':
            # Z-score method
            mean = df[column].mean()
            std_dev = df[column].std()
            z_scores = (df[column] - mean) / std_dev
            outliers = (z_scores.abs() > threshold)
        else:
            raise ValueError("Method must be 'iqr' or 'zscore'")
        
        outlier_counts[column] = int(outliers.sum())  # Count True values indicating outliers

    return outlier_counts

## DATA CLEANING PROCESS

Step 1. Import the needed packages

In [1]:
import pandas as pd
import numpy as np


Step 2. Load the dataset and display its first 10 rows.

In [8]:
df = pd.read_csv("../data/crypto_volatility_raw.csv")
df.head(10)

,timestamp,open,high,low,close,volume,bit_trend,fear&greed_index
0,2018-02-01,10285.10,10335.00,8750.99,9224.52,33564.764311,45,30.0
1,2018-02-02,9224.52,9250.00,8010.02,8873.03,49971.626975,65,15.0
2,2018-02-03,8873.03,9473.01,8229.00,9199.96,28725.000735,42,40.0
3,2018-02-04,9199.96,9368.00,7930.00,8184.81,32014.308449,34,24.0
4,2018-02-05,8179.99,8382.80,6625.00,6939.99,63403.182579,63,11.0
5,2018-02-06,6939.63,7878.00,6000.01,7652.14,100201.500307,78,8.0
6,2018-02-07,7655.02,8476.00,7150.01,7599.00,60778.460497,49,36.0
7,2018-02-08,7599.00,7844.00,7572.09,7784.02,1521.537318,41,30.0
8,2018-02-09,7789.90,8738.00,7789.90,8683.92,20482.910825,35,44.0
9,2018-02-10,8683.93,9065.78,8120.00,8533.98,49381.512653,28,54.0


Step 3. Check the df's size

In [9]:
df.shape

(2644, 8)

df has 8 columns (variables) and 2644 rows.

Step 4. Show the data type for each column

In [10]:
df.dtypes

timestamp            object
open                float64
high                float64
low                 float64
close               float64
volume              float64
bit_trend             int64
fear&greed_index    float64
dtype: object

Step 5. Change each column to its correct dtype  
Note. in this specific case, just "timestamp" will be changed

In [12]:
#change timestamp to datetime object
df["timestamp"] = pd.to_datetime(df["timestamp"])
#check it
df.dtypes

timestamp           datetime64[ns]
open                       float64
high                       float64
low                        float64
close                      float64
volume                     float64
bit_trend                    int64
fear&greed_index           float64
dtype: object

Step 6. Rename the column with easier names

In [14]:
#rename columns
df.rename(columns = {"timestamp" : "date",
                     "bit_trend" : "trend",
                     "fear&greed_index" : "fg_index"}, 
          inplace = True)
#check
df.head()

,date,open,high,low,close,volume,trend,fg_index
0,2018-02-01,10285.10,10335.00,8750.99,9224.52,33564.764311,45,30.0
1,2018-02-02,9224.52,9250.00,8010.02,8873.03,49971.626975,65,15.0
2,2018-02-03,8873.03,9473.01,8229.00,9199.96,28725.000735,42,40.0
3,2018-02-04,9199.96,9368.00,7930.00,8184.81,32014.308449,34,24.0
4,2018-02-05,8179.99,8382.80,6625.00,6939.99,63403.182579,63,11.0


Step 7. Look over missing values (NaN)

In [ ]:
#show sum of NaN in each column
df.isna().sum()

date        0
open        0
high        0
low         0
close       0
volume      0
trend       0
fg_index    0
dtype: int64

There are no missing values.

Step 8. Check for duplicates

In [17]:
find_duplicates(df)

,date,open,high,low,close,volume,trend,fg_index


As the function returns an empty df, there are not duplicates in our dataset. This means each date has been considered just once.

Step 9. Count outliers

In [23]:
#iqr method
count_outliers(df)

{'open': 12,
 'high': 12,
 'low': 18,
 'close': 12,
 'volume': 283,
 'trend': 66,
 'fg_index': 0}

In [ ]:
#Z-score method (how far away is the data from the mean?)
count_outliers(df, method = "zscore", threshold=3)
#thrheshold = 3 is a standard mesaures, it means the data is 3 standard deviations away from the mean

{'open': 0,
 'high': 1,
 'low': 1,
 'close': 0,
 'volume': 73,
 'trend': 28,
 'fg_index': 0}

At the moment, outliers will be kept as:  
  
- Are just a small % of the entire dataset
- Are actual information about Bitcoin volumes and trend.  
  
However, during the analysis the impact of outliers will be considered. 

Step 10. Save the cleaned dataset

In [26]:
#in csv
df.to_csv("../data/crypto_volatility_clean.csv", index = False)
#in parquet
df.to_parquet("../data/crypto_volatility_clean.parquet", index = False)